# ViT-LENS: Notebook 05 — Residual-Preserving Causal Intervention
**Research Goal**: Intervene directly inside the frozen ViT residual stream using learned TopK SAE feature directions:
$$x_{modified} = x + (f'_i - f_i) \cdot W_{dec}[i]$$
Test feature necessity via Suppression ($0\times$) and sufficiency via Amplification ($2\times$) with random feature control experiments.


In [ ]:
# Step 1: Initialize Causal Intervention System
import torch
from PIL import Image
from src.utils import get_device, load_config
from src.vit_model import ViTModelWrapper
from src.topk_sae import TopKSparseAutoencoder
from src.intervention import ResidualPreservingIntervention

device = get_device('auto')
config = load_config('config.yaml')

vit = ViTModelWrapper(pretrained=True, target_layer_idx=7, device=device)
sae = TopKSparseAutoencoder(d_in=768, d_sae=3072, top_k=32).to(device)
intervenor = ResidualPreservingIntervention(vit_wrapper=vit, sae=sae)


In [ ]:
# Step 2: Strict Numerical Verification of Identity Intervention (alpha = 1.0)
dummy_x = torch.randn(1, 3, 224, 224).to(device)
res_identity = intervenor.run_intervention(dummy_x, feature_id=127, strength_multiplier=1.0)

print(f'Logit Difference: {res_identity["logit_diff"]:.8f}')
print(f'Probability Difference: {res_identity["prob_diff"]:.8f}')
assert abs(res_identity['logit_diff']) < 1e-4, 'Identity intervention MUST yield zero difference!'


In [ ]:
# Step 3: Execute Suppression (0x) and Amplification (2x) Interventions
for strength in [0.0, 0.5, 1.0, 1.5, 2.0]:
    res = intervenor.run_intervention(dummy_x, feature_id=127, strength_multiplier=strength)
    print(f'Strength {strength:.1f}x -> Target Prob: {res["mod_target_prob"]:.4f} | '
          f'Prob Shift: {res["prob_diff"]:+.4f} | Flipped: {res["prediction_flipped"]}')
